## SPAD Experiment Tracking Automation

Runs a loop of alternating **position optimization** and **QM OPX+ sequence** passes using the SPC3 SPAD camera.

Each iteration:
1. **Laser ON** — `laser_switch_logic` set High
2. **Camera → alignment mode** + background correction from predefined `.spc3` file
3. **Position optimization** — Z sweep then XY sweep (`spad_optimize_logic`), scanner moved to peak
4. **Laser OFF** — `laser_switch_logic` set Low
5. **Camera → sequencing mode**
6. **Continuous acquisition** started — SPC3 writes data to disk
7. **AWG triggered** — `qm_switch_logic` pulsed High → Low; AWG runs OPX+ sequence and clocks camera
8. **Wait for AWG sequence to finish** — spc3 file size is monitored; stability → done
9. **Continuous acquisition stopped** → loop

**Prerequisites**
- qudi running with `SPUD202603.cfg`
- All modules active: `spad_optimize_logic`, `spad_probe_logic`, `camera_SPC3`, `camera_logic`,
  `laser_switch_logic`, `qm_switch_logic`
- Background `.spc3` snap file already captured (path set in **Configuration** cell)
- AWG script armed in triggered mode (waiting for `qm_switch_logic` rising edge)

In [ ]:
%matplotlib inline
import time
import os
import glob
import sys
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from pathlib import Path

## Qudi module handles

In [ ]:
opt       = spad_optimize_logic
spad      = spad_probe_logic
cam       = camera_SPC3
cam_logic = camera_logic
las_sw    = laser_switch_logic
qm_sw     = qm_switch_logic

# Derive switch key names from hardware (robust to config changes)
las_sw_key = list(las_sw.available_states.keys())[0]
qm_sw_key  = list(qm_sw.available_states.keys())[0]

print(f'Laser switch : {las_sw.device_name!r}  key={las_sw_key!r}  state={las_sw.get_state(las_sw_key)!r}')
print(f'QM switch    : {qm_sw.device_name!r}   key={qm_sw_key!r}   state={qm_sw.get_state(qm_sw_key)!r}')
print(f'Scanner pos  : {dict(spad.scanner_target)}')
print(f'Camera modes : {cam.get_available_modes()}')

## Configuration

In [ ]:
# Background SPC3 snap captured before this run (alignment mode)
BG_SPC3_FILE = r'C:\Users\SPUD1\Documents\experiment_workspace\SPAD data\spc3\spc3_snap_20260623-184203-136449.spc3'

# Directory where continuous-acquisition spc3 files will be written
DATA_DIR = r'C:\Users\SPUD1\Documents\experiment_workspace\SPAD data\spc3'

# Filename prefix for each iteration's spc3 file (timestamp + iteration number appended)
EXPERIMENT_TAG = 'ODMR_gradient_tracking'
#EXPERIMENT_NAME = 'ODMR_tracking'

# Total number of optimize-then-acquire iterations
N_ITERATIONS = 50

# Laser warm-up: wait this long after turning the laser on before running the
# position sweep (laser power takes ~20 s to stabilize)
LASER_SETTLE_S = 20

# AWG trigger: how long to hold qm_switch_logic High before pulling Low (seconds)
TRIGGER_HOLD_S = 5.0

# Experiment-done detection via spc3 file-size monitoring
STARTUP_TIMEOUT_S = 300  # max wait (s) for any data to appear; warns if exceeded
MIN_WAIT_S   = 60    # minimum rest after first data before quiescence check starts
POLL_S       = 10    # polling interval (s)
QUIESCENCE_S = 60    # file must be stable for this long to declare done
MAX_WAIT_S   = 7200  # hard timeout (2 h) for the quiescence phase

## Background image loading

In [ ]:
def _ensure_qudi_path():
    qudi_src = r'C:\Users\SPUD1\Documents\experiment_workspace\qudi-iqo-modules\src'
    if qudi_src not in sys.path:
        sys.path.insert(0, qudi_src)


def load_background(spc3_path):
    """Load a background SPC3 snap file.

    Returns
    -------
    bg_counts : (rows, cols) float32 — raw counts averaged over all frames
    bg_cps    : (rows, cols) float32 — counts per second
    header    : SPC3 file header
    """
    _ensure_qudi_path()
    from qudi.hardware.camera.SPC3.spc import SPC3 as SPC3SDK

    frames, header = SPC3SDK.ReadSPC3DataFile(str(spc3_path))
    ndim = getattr(frames, 'ndim', 0)
    if ndim == 4:
        seq = np.asarray(frames)[0]         # (N, rows, cols)
    elif ndim == 3:
        seq = np.asarray(frames)
    else:
        seq = np.asarray(frames)[None, ...]  # treat 2D as 1 frame

    bg_counts = seq.astype(np.float32).mean(axis=0)  # (rows, cols)

    # HwIntTime is the single-frame hardware integration time (seconds).
    # SummedFrames (= NIntegFrames) is the number of frames integrated per
    # output pixel.  The effective exposure per displayed frame is their product.
    hw_int_s = float(getattr(header, 'HwIntTime', 0.0) or 0.0)
    summed   = float(getattr(header, 'SummedFrames', 1.0) or 1.0)
    bg_exp_s = hw_int_s * summed if hw_int_s > 0 else 0.0
    bg_cps   = bg_counts / bg_exp_s if bg_exp_s > 0 else bg_counts.copy()

    return bg_counts, bg_cps, header


def enable_background_subtraction(bg_counts, bg_cps):
    """Enable background subtraction on the logic layer (snap/live) and
    optionally on the hardware layer (used by spad_optimize_logic)."""
    cam_logic.set_background_subtraction(True, bg_cps)
    setter = getattr(cam, 'set_background_subtraction_counts', None)
    if callable(setter):
        setter(True, bg_counts)


def disable_background_subtraction():
    cam_logic.set_background_subtraction(False)
    setter = getattr(cam, 'set_background_subtraction_counts', None)
    if callable(setter):
        setter(False, None)

## Position optimization helpers

Adapted from `SPAD_optimize_sweep.ipynb` — runs Z then XY sweep without interactive plots.

In [ ]:
def _moving_avg(y, w):
    w = max(1, int(w))
    if w % 2 == 0:
        w += 1
    if w == 1:
        return y.astype(float)
    pad = w // 2
    return np.convolve(
        np.pad(y.astype(float), pad, mode='edge'),
        np.ones(w) / w,
        mode='valid'
    )


def _peak_z_um(z_um, y):
    """Robust Z-peak finder: smoothed argmax + local quadratic fit."""
    z, y = np.asarray(z_um, float), np.asarray(y, float)
    if z.size < 3:
        return float(z[np.argmax(y)])
    w = max(5, min(21, round(0.07 * z.size)))
    if w % 2 == 0:
        w += 1
    ys = _moving_avg(y, w)
    ci = int(np.argmax(ys))
    i0, i1 = max(0, ci - 6), min(z.size - 1, ci + 6)
    if i1 - i0 < 2:
        return float(z[ci])
    p2, p1, _ = np.polyfit(z[i0:i1 + 1], y[i0:i1 + 1], 2)
    if np.isfinite(p2) and p2 < 0:
        return float(np.clip(-p1 / (2 * p2), z[i0], z[i1]))
    return float(z[ci])


def _bright9(frames_nhw):
    """Sum of 9 brightest pixels per frame.  frames_nhw: (N, H, W) -> (N,)"""
    flat = frames_nhw.reshape(frames_nhw.shape[0], -1)
    k = min(9, flat.shape[1])
    return np.partition(flat, -k, axis=1)[:, -k:].sum(axis=1)


def _axis_key(pos_dict, name):
    return next(k for k in pos_dict if str(k).lower() == name.lower())


def run_optimize_sweep(iteration=None):
    """Run Z then XY optimization sweep, plot results, and move scanner to peak."""
    tag = f'iter {iteration}' if iteration is not None else ''

    # ── Z sweep ────────────────────────────────────────────────────────
    opt.scan_sequence = (('z',),)
    opt.optimizer_sequence_dimensions = (1,)
    opt.start_optimize()
    t0 = time.time()
    while opt.optimizer_running:
        time.sleep(0.5)
    print(f'    Z sweep complete in {time.time() - t0:.0f}s')

    z_data   = opt.spad_scan_data
    z_ax     = tuple(z_data[0]['axes'])[0]
    z_um     = np.array(z_data[0]['grids'][z_ax]) * 1e6   # m -> µm
    frames_z = np.array(z_data[0]['frames'])               # (Nz, H, W)
    signal_z = _bright9(frames_z)
    z_peak_um = _peak_z_um(z_um, signal_z)

    # Z sweep plot
    w = max(5, min(21, round(0.07 * z_um.size)))
    if w % 2 == 0:
        w += 1
    ys = _moving_avg(signal_z, w)
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(z_um, signal_z, 'o', ms=3, alpha=0.6, label='data')
    ax.plot(z_um, ys, '-', lw=2, label=f'smoothed (MA w={w})')
    ax.axvline(z_peak_um, color='r', ls='--', lw=1.5, label=f'peak {z_peak_um:.3f} µm')
    ax.set_xlabel('Z (µm)')
    ax.set_ylabel('9-brightest sum (counts)')
    ax.set_title(f'Z sweep{"  —  " + tag if tag else ""}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    pos = dict(spad.scanner_target)
    pos[_axis_key(pos, 'z')] = z_peak_um * 1e-6
    spad.set_target_position(pos, move_blocking=True)
    print(f'    Z -> {z_peak_um:.3f} um')

    # ── XY sweep ───────────────────────────────────────────────────────
    opt.scan_sequence = (('x', 'y'),)
    opt.optimizer_sequence_dimensions = (2,)
    opt.start_optimize()
    t0 = time.time()
    while opt.optimizer_running:
        time.sleep(0.5)
    print(f'    XY sweep complete in {time.time() - t0:.0f}s')

    xy_data  = opt.spad_scan_data
    ax0, ax1 = tuple(xy_data[0]['axes'])
    g0 = np.array(xy_data[0]['grids'][ax0]) * 1e6   # µm
    g1 = np.array(xy_data[0]['grids'][ax1]) * 1e6
    frames_xy = np.array(xy_data[0]['frames'])        # (Nx, Ny, H, W)
    nx, ny = len(g0), len(g1)

    flat_frames = frames_xy.reshape(nx * ny, *frames_xy.shape[2:])
    metric = _bright9(flat_frames).reshape(nx, ny)
    pi, pj = np.unravel_index(np.nanargmax(metric), metric.shape)

    # XY sweep plot
    fig, ax = plt.subplots(figsize=(5, 4))
    extent = [g0[0], g0[-1], g1[0], g1[-1]]
    ax.imshow(metric.T, origin='lower', aspect='auto', extent=extent, cmap='inferno')
    ax.plot(g0[pi], g1[pj], 'w+', ms=12, mew=2,
            label=f'peak ({g0[pi]:.3f}, {g1[pj]:.3f}) µm')
    ax.set_xlabel(f'{ax0} (µm)')
    ax.set_ylabel(f'{ax1} (µm)')
    ax.set_title(f'XY sweep{"  —  " + tag if tag else ""}')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    pos = dict(spad.scanner_target)
    pos[_axis_key(pos, ax0)] = float(g0[pi]) * 1e-6
    pos[_axis_key(pos, ax1)] = float(g1[pj]) * 1e-6
    spad.set_target_position(pos, move_blocking=True)
    print(f'    XY -> ({g0[pi]:.3f}, {g1[pj]:.3f}) um')


    # Averaged frame at the optimized XY position
    opt_frame = frames_xy[pi, pj].copy()
    # print(f'    Averaged frame: shape={opt_frame.shape}, '
    #       f'min={opt_frame.min():.1f}, max={opt_frame.max():.1f}, '
    #       f'sum={opt_frame.sum():.1f} counts, NFrames={cam._NFrames}')
    fig_f, ax_f = plt.subplots(figsize=(5, 4))
    im_f = ax_f.imshow(opt_frame, origin='lower', cmap='inferno')
    _title = (f'Averaged frame at optimized position\n'
              f'{ax0}={g0[pi]:.3f} µm, {ax1}={g1[pj]:.3f} µm'
              + (f'  —  {tag}' if tag else ''))
    ax_f.set_title(_title)
    ax_f.set_xlabel('pixel column')
    ax_f.set_ylabel('pixel row')
    fig_f.colorbar(im_f, ax=ax_f, label='counts')
    plt.tight_layout()
    plt.show()
    final = {k: f'{v * 1e6:.3f} um' for k, v in dict(spad.scanner_target).items()}
    print(f'    Final position: {final}')

## Experiment-done detection

When the AWG sequence finishes, the SPC3 camera stops receiving triggers and stops writing frames.
The spc3 file size becomes stable; we use that as the completion signal.

In [ ]:
def _spc3_total_bytes(stem):
    """Total bytes across all spc3 split-files for one acquisition stem.

    The SPC3 SDK writes stem.spc3, stem_shift1.spc3, stem_shift2.spc3 …
    when a file reaches its size limit.
    """
    d = os.path.dirname(stem)
    b = os.path.basename(stem)
    files = glob.glob(os.path.join(d, b + '*.spc3'))
    return sum(os.path.getsize(p) for p in files if os.path.exists(p))


def wait_for_acquisition_done(stem,
                              startup_timeout_s=STARTUP_TIMEOUT_S,
                              min_wait_s=MIN_WAIT_S,
                              quiescence_s=QUIESCENCE_S,
                              poll_s=POLL_S,
                              max_wait_s=MAX_WAIT_S):
    """Wait until the spc3 file stops growing (AWG sequence done).

    Two-phase:
      Phase 1 — startup: wait until the file grows beyond the initial header
                patch written by _patch_cont_files_inplace (~1 kB). If no
                photon data arrives within startup_timeout_s, print a warning
                and return early.
      Phase 2 — quiescence: wait until file size is stable for quiescence_s s.

    Data flushing (ContAcqToFileGetMemory) is handled automatically by the
    camera_logic Qt timer, which now starts correctly on the logic thread via
    QMetaObject.invokeMethod in camera_logic_SPC3.py.
    """
    t0 = time.time()
    HEADER_BYTES = 4096  # initial header written by _patch_cont_files_inplace

    # ── Phase 1: wait for photon data beyond header ────────────────────────────
    print(f'    Phase 1: waiting for data beyond header (timeout {startup_timeout_s:.0f}s) ...')
    while True:
        time.sleep(poll_s)
        elapsed = time.time() - t0
        size = _spc3_total_bytes(stem)
        if size > HEADER_BYTES:
            print(f'    First photon data: {size / 1e3:.1f} kB at {elapsed:.0f}s')
            break
        if elapsed >= startup_timeout_s:
            print(f'    WARNING: no photon data received in {elapsed:.0f}s (file={size} B).')
            print(f'    Check: AWG sequence armed, camera SYNC-IN connected.')
            return
        print(f'    {elapsed:.0f}s | {size} B | waiting for triggers ...')

    # ── Phase 2: minimum rest, then quiescence monitoring ─────────────────────
    rest = min_wait_s - (time.time() - t0)
    if rest > 0:
        print(f'    Resting {rest:.0f}s before quiescence monitoring ...')
        time.sleep(rest)

    last_size = _spc3_total_bytes(stem)
    stable_t  = time.time()

    while True:
        time.sleep(poll_s)
        elapsed = time.time() - t0

        if elapsed > max_wait_s:
            print(f'    Hard timeout ({elapsed / 60:.1f} min). Proceeding.')
            return

        size = _spc3_total_bytes(stem)
        if size != last_size:
            last_size = size
            stable_t  = time.time()

        stable_for = time.time() - stable_t
        print(f'    {elapsed / 60:.1f} min | {size / 1e6:.2f} MB | '
              f'stable {stable_for:.0f}s / {quiescence_s}s')

        if stable_for >= quiescence_s:
            print(f'    Acquisition done: {size / 1e6:.2f} MB total.')
            return

## Pre-loop: load background image

Done once before the loop starts so the same background is reused across all iterations.

In [ ]:
print('Loading background image ...')
bg_counts, bg_cps, bg_header = load_background(BG_SPC3_FILE)

hw_int_s = getattr(bg_header, 'HwIntTime', None)
summed   = getattr(bg_header, 'SummedFrames', None)
bg_exp_s = (hw_int_s * summed) if (hw_int_s and summed) else None

print(f'  shape          : {bg_counts.shape}')
print(f'  mean counts    : {bg_counts.mean():.2f}')
print(f'  HwIntTime      : {hw_int_s} s  (single-frame HIT)')
print(f'  SummedFrames   : {summed}  (NIntegFrames)')
print(f'  effective exp  : {bg_exp_s} s  (HwIntTime x SummedFrames)')
print(f'  mean bg cps    : {bg_cps.mean():.1f}')

os.makedirs(DATA_DIR, exist_ok=True)

## Main experiment loop

The `finally` block guarantees the laser and AWG trigger are left Low and continuous acquisition
is stopped even if the loop is interrupted.

In [ ]:
try:
    for k in range(N_ITERATIONS):
        t_iter = time.time()
        ts = datetime.now().strftime('%Y%m%d-%H%M%S')
        print(f'\n=== Iteration {k + 1}/{N_ITERATIONS}  [{ts}] ===')

        # ── 1. Laser ON ───────────────────────────────────────────────
        print('  1. Laser ON')
        las_sw.set_state(las_sw_key, 'High')

        # ── 2. Camera -> alignment mode + background correction ───────
        print('  2. Camera -> alignment mode; background correction enabled')
        cam.set_mode('alignment')
        enable_background_subtraction(bg_counts, bg_cps)
        print(f'     Waiting {LASER_SETTLE_S}s for laser power to stabilize ...')
        time.sleep(LASER_SETTLE_S)

        # ── 3. Position optimization ──────────────────────────────────
        print('  3. Running position optimization (Z then XY) ...')
        run_optimize_sweep(iteration=k + 1)

        # ── 4. Laser OFF ──────────────────────────────────────────────
        print('  4. Laser OFF')
        las_sw.set_state(las_sw_key, 'Low')
        disable_background_subtraction()

        # ── 5. Camera -> sequencing mode ──────────────────────────────
        # set_mode() commits all hardware settings (HIT, NInteg, trigger, gate)
        # via a single SetCameraPar + ApplySettings sequence.  Do NOT call
        # set_trigger_mode() or any other set_* method after this — a second
        # ApplySettings() immediately before ContAcqToFileStart resets the
        # camera into an idle state that prevents data generation.
        print('  5. Camera -> sequencing mode')
        cam.set_mode('sequencing')

        # ── 6. Start continuous acquisition ───────────────────────────
        ts_acq = datetime.now().strftime('%Y%m%d-%H%M%S-%f')
        stem = os.path.join(DATA_DIR, f'{EXPERIMENT_TAG}_{ts_acq}_iter{k + 1:03d}')
        print(f'  6. Starting continuous acquisition')
        print(f'     stem: {os.path.basename(stem)}')
        acq_path = cam_logic.start_continuous(stem)
        if not acq_path:
            print('  ERROR: Failed to start continuous acquisition. Aborting.')
            break

        # ── 7. Trigger AWG ────────────────────────────────────────────
        print(f'  7. Triggering AWG (High for {TRIGGER_HOLD_S:.1f}s then Low)')
        qm_sw.set_state(qm_sw_key, 'High')
        time.sleep(TRIGGER_HOLD_S)
        qm_sw.set_state(qm_sw_key, 'Low')

        # ── 8. Wait for AWG sequence to finish ────────────────────────
        # Data flushing (ContAcqToFileGetMemory) is driven by the camera_logic
        # Qt timer (100 ms interval), which now starts on the correct thread via
        # QMetaObject.invokeMethod — no manual pumping needed from the notebook.
        print('  8. Waiting for AWG sequence to complete (file-size monitor) ...')
        wait_for_acquisition_done(stem)

        # ── 9. Stop continuous acquisition ────────────────────────────
        print('  9. Stopping continuous acquisition')
        cam_logic.stop_continuous()

        print(f'  Iteration {k + 1} done in {(time.time() - t_iter) / 60:.1f} min')

finally:
    # Safety cleanup — always runs, even on KeyboardInterrupt
    las_sw.set_state(las_sw_key, 'Low')
    qm_sw.set_state(qm_sw_key, 'Low')
    if cam_logic.continuous_active:
        cam_logic.stop_continuous()
    disable_background_subtraction()
    print('\nCleanup complete (laser Low, AWG trigger Low, acquisition stopped).')

print(f'\nAll {N_ITERATIONS} iterations complete.')